# Программирование для всех<br>(основы работы с Python)

*Тамбовцева А.А.*

## Практикум 6.1. Работа с API на примере ВКонтакте: извлечение постов со страницы сообщества

Загрузим необходимые для работы библиотеки, модули и функции:

* модуль `requests` для отправки запроса к серверу и получения ответа;
* библиотека `pandas` для работы с таблицами;
* функция `getpass()` из модуля `getpass` для более «секретного» ввода с клавиатуры.

In [ ]:
import requests
import pandas as pd
from getpass import getpass

Ранее мы получили доступ к API ВКонтакте через **сервисный ключ** приложения (если нет – см инструкцию [здесь](https://github.com/allatambov/PyAll26/blob/main/vk-api-access.md)). Этот сервисный ключ можно скопировать в строку ввода данных ниже:

In [ ]:
# getpass() как input(), только при вводе данных
# не показывает вводимый текст на экране

service_key = getpass()

В [документации](https://dev.vk.com/ru/reference) API зайдем в раздел «Формат запросов» и изучим синтаксис запроса, чтобы понять код ниже и структуру ссылки в `res.url`.

In [ ]:
main_url = "https://api.vk.ru/method/wall.get"

wall_params = {
    "domain": "hseteachers", 
    "count": 100, 
    "filter" : "owner", 
    "access_token": service_key, 
    "v": "5.199"
}

In [ ]:
res = requests.get(main_url, params = wall_params)

Объект, который мы сохранили в `res`, имеет особый тип `requests.models.Response`, он же ответ на запрос:

In [ ]:
print(type(res))
print(res)

Сам по себе объект полностью мы увидеть не можем, но из него можно извлечь разные атрибуты – характеристики объекта. Например, статус ответа:

In [ ]:
print(res.status_code)

> Код 200 означает, что запрос получен, обработан, и ответ на него получен. Другими словами, у нас нет проблем с доступом к странице (потенциальная проблема со стороны сервера, например, ведущиеся технические работы или отказ в правах доступа) и нет проблем с самим запросом (потенциальная проблема со стороны пользователя, например, «битая» ссылка). Про коды статусов можно почитать в [статье](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).

Или ссылку, которая была сформирована функцией `get()` для отправки запроса:

In [ ]:
print(res.url)

Если перейти по ссылке, откроется страница в браузере с огромным полотном текста – это ответ на запрос в виде большой JSON-строки. Если, используя Python, мы заберем текст «как есть», придется долго ломать голову, как из такого массива символов выбрать нужное:

In [ ]:
# плохая идея
# print(res.text)

А вот преобразовать эту JSON-строку через метод `.json()` в структуру данных Python – идея хорошая:

In [ ]:
res_dict = res.json()

## Сюжет 1. Забираем 100 постов со стены сообщества и очищаем данные

Выше мы преобразовали JSON-строку с ответом на запрос в один большой словарь (десериализовали JSON в Python). Теперь мы можем поработать с этим словарем и извлечь из него необходимые данные!

### Задача 1

Получите, используя словарь `res_dict`, список словарей `items`, где один словарь соответствует одному посту.  

In [ ]:
### YOUR CODE HERE ###

### Задача 2

Напишите функцию `clean_posts()`, которая принимает на вход один словарь с характеристиками поста и возвращает список из следующих значений:

* id поста
* дата публикации 
* текст поста
* число лайков
* число просмотров

Примените функцию к каким-нибудь элементам в `items` и проверьте, что она работает. Как вы думаете, может ли у поста какая-то характеристика отсутствовать? Как скорректировать функцию, чтобы учесть такой сценарий?

In [ ]:
### YOUR CODE HERE ###

### Задача 3

Напишите код (используя функцию из задачи 2), который сохранит в список `final` характеристики только тех постов, которые не содержат прикрепленных изображений и иных файлов.

In [ ]:
### YOUR CODE HERE ###

Итак, если `final` содержит списки с характеристиками постов (один список = один пост), его можно преобразовать в таблицу, объект типа `pandas DataFrame` (один список = одна строка в таблице):

In [ ]:
df = pd.DataFrame(final, columns = ["id", "date", "text", "likes", "views"])
df

Преобразуем дату в более привычный формат:

In [ ]:
df["date"] = pd.to_datetime(df["date"], unit="s")
df

Можем выгрузить в CSV-файл или файл Excel:

In [ ]:
df.to_csv("hseposts_100.csv")
df.to_excel("hseposts_100.xlsx")

### Сюжет 2. Забираем все посты со стены сообщества, создаем и обрабатываем таблицу

Ограничения данного API таковы, что за один раз мы можем выгрузить только 100 постов. Хорошие новости: каждый раз при выгрузке мы можем начинать с того поста, на котором закончили, то есть сначала выгрузить первые 100 постов, потом – следующие 100 постов, и так до тех пор, пока не заполучим все. Обеспечивается это аргументом `offset` в методе `wall.get` в API.

Тем не менее, остаются ограничения на число запросов в секунду. Для надежности (во избежание блокировки) импортируем функцию `sleep` из модуля `time`, которая позволит выставлять задержку в одну секунду после выгрузки очередных 100 постов.

In [ ]:
from time import sleep

### Задача 1

Сохраните общее число постов в переменную `nposts`. Посчитайте, сколько раз нужно будет выполнить выгрузку по 100 постов, чтобы собрать все тексты, и сохраните значение в переменную `iterate`.

In [ ]:
### YOUR CODE HERE ###

### Задача 2

Используя информацию об аргументе `offset` и блоки кода ниже, выгрузите и сохраните в список `items_all` данные по всем постам на стене сообщества.

**Подсказка:** чтобы расширять список правильным образом, используйте метод `.extend()`, а не `.append()`, он добавляет не один элемент, а сразу несколько.

In [ ]:
wall_params_long = {
    "domain": "hseteachers", 
    "count": 100, 
    "offset" : 0,
    "filter" : "owner", 
    "access_token": service_key, 
    "v": "5.199"
}

items_all = []

for i in range(iterate):
    res_long = requests.get(main_url, params = wall_params_long)
    res_dict_long = res_long.json()
    data_long = res_dict_long['response']['items']
    
    ### YOUR CODE HERE ###
    items_all.extend(data_long)
    
    sleep(1)
    print(i, end = " ... ")

In [ ]:
print(len(items_all))

Ниже написан код, который переводит список `items_all` в датафрейм `df_full`. Так как здесь данные мы предварительно не очищали и не фильтровали, столбцов много, и некоторые из них со словарями внутри. Выберем из `full` только нужные столбцы и сохраним в `df_small`.

In [ ]:
df_full = pd.DataFrame(items_all)
df_small = df_full[["id", "date", "text", "likes", "views"]]
df_small

Осталось разобраться со словарями внутри ячеек таблицы!

### Задача 3

В `pandas` есть метод `.apply()`, который позволяет применить какую-то функцию ко всем ячейкам в столбце.

In [ ]:
# применяем ко всем ячейкам в столбце функцию len(),
# считаем количество символов

df_small["text"].apply(len)

Его удобно сочетать с короткими анонимными lambda-функциями:

In [ ]:
# применяем ко всем ячейкам в столбце lambda-функцию,
# разбиваем на «слова» и считаем количество слов

df_small["text"].apply(lambda x: len(x.split()))

Используя метод `.apply()` и lambda-функции, извлеките из `likes` и `views` число лайков и просмотров и сохраните их в столбцы `nlikes` и `nviews`. 

In [ ]:
### YOUR CODE HERE ###